# Re-run the conditional-expert drug-response study

A thin wrapper around the same scripts the study ran. It calls them; it does not reimplement
anything, and it is not where the research happened — see `reports/moe_drug_v1/RESULTS.md`
for what was found.

Everything runs through Slurm from the G1 worktree root. The large inputs stay on the server:
this notebook reads the small manifests and result tables that are in Git.

**Order matters.** The test split is opened by the last step only, after the models are frozen.
Running step 6 before step 5 would put test scores inside the selection procedure.

In [ ]:
ROOT = "/mnt/S1/sdata/workbench/Virtual_Cell/gssong/vcell_liver_conditional_moe_v1"
COMMIT = "research/conditional-moe-drug-ood-v1"
print(f"cd {ROOT}  # git checkout {COMMIT}")

## The whole pipeline, as submitted

```bash
cd $ROOT

# 1. build the frozen (cell line, drug) table from the reused R3 aggregation   (~26 s, CPU)
P=$(sbatch --parsable experiments/moe_drug_v1/prepare.sbatch)

# 2. the four pre-run checks; everything below depends on these passing        (~15 s, 1 GPU)
K=$(sbatch --parsable --dependency=afterok:$P experiments/moe_drug_v1/checks.sbatch)

# 3. minimum baselines on validation                                           (~10 s, 1 GPU)
sbatch --dependency=afterok:$K experiments/moe_drug_v1/baselines.sbatch val
sbatch --dependency=afterok:$K experiments/moe_drug_v1/describe.sbatch

# 4. the identical six-setting grid for each model, seed 42                    (~2 min each, 1 GPU)
sbatch --dependency=afterok:$K experiments/moe_drug_v1/sweep.sbatch A 42
sbatch --dependency=afterok:$K experiments/moe_drug_v1/sweep.sbatch C 42

# 5. the selected setting at the two confirmation seeds, then selection + gate
for S in 43 44; do for M in A C; do
  sbatch experiments/moe_drug_v1/train.sbatch $M 3e-3 1e-2 $S confirm
done; done
sbatch experiments/moe_drug_v1/aggregate.sbatch

# 6. B is trained only if reports/moe_drug_v1/selection.json says train_B: true
#    sbatch experiments/moe_drug_v1/sweep.sbatch B 42

# 7. freeze, then open the test split once for every model with one code path
sbatch experiments/moe_drug_v1/final_eval.sbatch
```

The selected learning rate and weight decay in step 5 are whatever step 4 chose on validation;
`reports/moe_drug_v1/selection.json` records them. Do not substitute values from the report
without re-running step 4.

## Read the results that are in Git

In [ ]:
import json
import pandas as pd

R = "reports/moe_drug_v1"
trials = pd.read_csv(f"{R}/trials.tsv", sep="\t")
trials[["run", "kind", "lr", "wd", "seed", "val_macro_delta_rmse", "best_epoch"]]

In [ ]:
sel = json.load(open(f"{R}/selection.json"))
print("selected:", json.dumps(sel["selected"], indent=1))
print("train B?", sel["train_B"], "| mean C-over-A improvement %:",
      round(sel["mean_relative_improvement_pct"], 4),
      "| threshold %:", sel["threshold_pct"])
print("beats baselines:", sel["beats_baselines"])

In [ ]:
test = json.load(open(f"{R}/test_results.json"))
pd.DataFrame(test["summary"]).T[["macro_delta_rmse", "drug_centred_r",
                                 "top20_direction", "snu423_delta_rmse"]]

In [ ]:
b = test["bootstrap"]["A_vs_C"]
print(f"A - C = {b['diff']:+.5f}  95% CI [{b['diff_ci'][0]:+.5f}, {b['diff_ci'][1]:+.5f}]")
print(f"relative {b['rel_improvement_pct']:+.3f}%  "
      f"CI [{b['rel_improvement_ci_pct'][0]:+.3f}%, {b['rel_improvement_ci_pct'][1]:+.3f}%]")
print("resampling unit: drug group, carrying all its cell lines together;")
print("conditional on these trained models, not a statement about training randomness.")

## What this notebook is not

It does not re-select anything, re-fit a calibrator on test truth, or make a figure that was
not already in the reports. If you change a setting here, you are starting a new study and it
needs its own pre-registration and its own held-out split.